In [19]:
import pandas as pd
from datetime import datetime 
import streamlit as st

In [20]:
st.set_page_config(
    page_title="Electronic Trading Risk | Drop-Copy Reconciliation",
    page_icon="⚡",
    layout="wide"
)

2026-10-12 17:11:51.472 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [21]:
@st.cache_data
def generate_mock_data():
    """
    Simulates internal OMS blotter records vs. Exchange Drop-Copy (FIX execution reports).
    We intentionally inject discrepancy types: Match, Price Break, Orphan Order, and Execution Drop.
    """

    oms_data = [
        {"ClOrdID": "ORD-1001", "Symbol": "AAPL", "Side": "BUY",  "Qty": 500, "Price": 180.50, "Status": "FILLED"},
        {"ClOrdID": "ORD-1002", "Symbol": "MSFT", "Side": "SELL", "Qty": 200, "Price": 415.00, "Status": "FILLED"},
        {"ClOrdID": "ORD-1003", "Symbol": "NVDA", "Side": "BUY",  "Qty": 1000,"Price": 875.25, "Status": "PARTIAL"},
        {"ClOrdID": "ORD-1004", "Symbol": "AMZN", "Side": "BUY",  "Qty": 300, "Price": 178.00, "Status": "NEW"},     # Missing drop-copy (Execution Drop)[cite: 1]
    ]
    dropcopy_data = [
        {"ClOrdID": "ORD-1001", "ExecID": "EX-9001", "Symbol": "AAPL", "Side": "BUY",  "LastQty": 500, "LastPx": 180.50, "ExecType": "Fill"}, # Perfect Match
        {"ClOrdID": "ORD-1002", "ExecID": "EX-9002", "Symbol": "MSFT", "Side": "SELL", "LastQty": 200, "LastPx": 415.50, "ExecType": "Fill"}, # Price Break (OMS: 415.00 vs Exch: 415.50)[cite: 1]
        {"ClOrdID": "ORD-1003", "ExecID": "EX-9003", "Symbol": "NVDA", "Side": "BUY",  "LastQty": 1000,"LastPx": 875.25, "ExecType": "Fill"}, # Variance check
        {"ClOrdID": "ORD-9999", "ExecID": "EX-9999", "Symbol": "GOOGL","Side": "BUY",  "LastQty": 150, "LastPx": 140.20, "ExecType": "Fill"}, # Orphan Order (Not in OMS)[cite: 1]
    ]
    return pd.DataFrame(oms_data), pd.DataFrame(dropcopy_data)

2026-10-12 17:11:51.500 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [22]:
# ==========================================
# RECONCILIATION ENGINE LOGIC
# ==========================================

In [23]:
def run_reconciliation(oms_df, dc_df):
    merged = pd.merge(oms_df, dc_df, on="ClOrdID", how="outer", suffixes=("_OMS", "_EXCH"))
    reconciliation_results = []

In [24]:
def run_reconciliation(oms_df, dc_df):
    merged = pd.merge(oms_df, dc_df, on="ClOrdID", how="outer", suffixes=("_OMS", "_EXCH"))
    reconciliation_results = []
    
    for _, row in merged.iterrows():
        cl_ord_id = row["ClOrdID"]
        
        if pd.isna(row["Symbol_OMS"]):
            reconciliation_results.append({
                "ClOrdID": cl_ord_id,
                "Symbol": row["Symbol_EXCH"],
                "Category": "ORPHAN ORDER",
                "Severity": "HIGH",
                "Details": f"Exchange executed ExecID {row['ExecID']} but no order found in internal OMS blotter."
            })
            continue
            
        if pd.isna(row["ExecID"]):
            reconciliation_results.append({
                "ClOrdID": cl_ord_id,
                "Symbol": row["Symbol_OMS"],
                "Category": "EXECUTION DROP",
                "Severity": "CRITICAL",
                "Details": "OMS shows order active/submitted, but no exchange drop-copy fill confirmation received."
            })
            continue
            
        price_diff = abs(row["Price"] - row["LastPx"])
        if price_diff > 0.01:
            reconciliation_results.append({
                "ClOrdID": cl_ord_id,
                "Symbol": row["Symbol_OMS"],
                "Category": "PRICE TRADE BREAK",
                "Severity": "MEDIUM",
                "Details": f"Price Mismatch! OMS: {row['Price']} vs Exchange: {row['LastPx']}"
            })
        else:
            reconciliation_results.append({
                "ClOrdID": cl_ord_id,
                "Symbol": row["Symbol_OMS"],
                "Category": "MATCHED",
                "Severity": "NONE",
                "Details": "Trade verified successfully."
            })
            
    return pd.DataFrame(reconciliation_results)

In [25]:
git remote add origin https://github.com/karthikdas123/fix-drop-copy-engine.git
git branch -M main
git push -u origin main


SyntaxError: invalid syntax (3091683506.py, line 1)